# ⚡ Z-Fooocus
**Multi-Model Studio** — Generate · Img2Img · Inpaint with model switching

Select which models you want below, then run **Step 1** to install & download.

In [ ]:
# @title 🎯 Select Models to Download
# @markdown **Pick the models you need** (uncheck to skip download)
# @markdown ---
Z_Image_Turbo = True  # @param {type:"boolean"}
# @markdown ⚡ **Z-Image Turbo** — Fast generation only (8 steps, ~6 GB)
FLUX_Klein_4B = True  # @param {type:"boolean"}
# @markdown 🌊 **FLUX.2-klein 4B** — Generation, Img2Img, Inpaint (~7.8 GB + 3.85 GB encoder)
FLUX_Klein_9B = False  # @param {type:"boolean"}
# @markdown 🔮 **FLUX.2-klein 9B** — Higher quality, may OOM on T4 (~10 GB + 6.8 GB encoder)
Qwen_Image_Edit = True  # @param {type:"boolean"}
# @markdown 🎨 **Qwen-Image-Edit-2511** — Instruction-based Img2Img & Inpaint (~13 GB GGUF)

In [ ]:
# ── Step 1: Install & Download Selected Models ─────────────
import os

# ── ComfyUI ───────────────────────────────────────────────
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

# ComfyUI-GGUF custom nodes (needed for Qwen-Image-Edit)
if Qwen_Image_Edit and not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

!pip install -q rembg[gpu] onnxruntime-gpu
!pip install -q aria2

# ── Model dirs ────────────────────────────────────────────
DIFF    = '/content/ComfyUI/models/diffusion_models'
CLIP    = '/content/ComfyUI/models/clip'
TXTENC  = '/content/ComfyUI/models/text_encoders'
VAE     = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if os.path.exists(path):
        print(f'  ✅ {name}')
    else:
        print(f'  ⏳ {name}...')
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url} -d {dest_dir} -o {name}
        print(f'  ✅ {name}')

# ── Z-Image Turbo ─────────────────────────────────────────
if Z_Image_Turbo:
    print('\n📦 Z-Image Turbo FP8')
    dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

# ── FLUX.2 shared VAE (needed by both 9B and 4B) ──────────
if FLUX_Klein_9B or FLUX_Klein_4B:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

# ── FLUX.2-klein 4B ───────────────────────────────────────
if FLUX_Klein_4B:
    print('\n📦 FLUX.2-klein 4B')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors', DIFF, 'flux-2-klein-4b.safetensors')
    # FLUX.2-specific Qwen3 4B encoder (7680-dim, NOT the Z-Image 2560-dim version!)
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors', TXTENC, 'qwen_3_4b_fp4_flux2.safetensors')

# ── FLUX.2-klein 9B FP8 ───────────────────────────────────
if FLUX_Klein_9B:
    print('\n📦 FLUX.2-klein 9B FP8 (⚠️ may OOM on T4)')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors', DIFF, 'flux-2-klein-9b-kv-fp8.safetensors')
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/text_encoders/qwen_3_8b_fp4mixed.safetensors', TXTENC, 'qwen_3_8b_fp4mixed.safetensors')

# ── Qwen-Image-Edit-2511 GGUF Q4_K_M ──────────────────────
if Qwen_Image_Edit:
    print('\n📦 Qwen-Image-Edit-2511 GGUF Q4_K_M')
    dl('https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q4_K_M.gguf', DIFF, 'qwen-image-edit-2511-Q4_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf')
    dl('https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

# Symlink clip/ → text_encoders/ (ComfyUI CLIPLoader searches text_encoders/)
import glob
for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)

print('\n🎉 Selected models ready!')

In [ ]:
# ── Step 2: Launch Z-Fooocus ──────────────────────────────
import os
os.chdir('/content')
REPO = 'https://github.com/MuntasirMalek/Z-Fooocus.git'
if os.path.exists('/content/Z-Fooocus'):
    %cd /content/Z-Fooocus
    !git pull --ff-only || true
else:
    !git clone {REPO}
    %cd /content/Z-Fooocus
!python app.py